In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

# -------------------------------------------------------------------
# 1. Загрузка данных
# -------------------------------------------------------------------
import os
METRICS_PATH = "recomputed_metrics.pkl" if os.path.exists("recomputed_metrics.pkl") else "scaling/recomputed_metrics.pkl"
with open(METRICS_PATH, "rb") as f:
    results = pickle.load(f)

data = results["val_loss"]
dead_neurons_data = results["dead_proportion"]
active_per_token_data = results["active_per_token"]

mlp_mults = sorted(set(m for (m, _) in data.keys()))
topks = sorted(set(k for (_, k) in data.keys()), key=lambda x: (x is None, x))

# -------------------------------------------------------------------
# 2. Вспомогательные функции для осветления/затемнения цвета
# -------------------------------------------------------------------
def lighten_color(color, factor=0.5):
    """Осветляет color (factor=0 -> исходный, 1 -> белый)."""
    rgb = np.array(mcolors.to_rgb(color))
    white = np.array([1, 1, 1])
    return mcolors.to_hex(rgb + (white - rgb) * factor)

def darken_color(color, factor=0.5):
    """Затемняет color (factor=0 -> исходный, 1 -> чёрный)."""
    rgb = np.array(mcolors.to_rgb(color))
    black = np.array([0, 0, 0])
    return mcolors.to_hex(rgb + (black - rgb) * factor)

base_red = "#8a3434"
red_pink_cmap = LinearSegmentedColormap.from_list(
    "red_pink",
    [darken_color(base_red, 0.7), base_red, lighten_color(base_red, 0.8)],
    N=256
)

palette = [
    "#5f5a4a", "#6f6754", "#7f745e", "#8f8068", "#9f8d72",
    "#8a3434", "#984444", "#a65454", "#b46464", "#c27474",
]

# -------------------------------------------------------------------
# Plot 1: Heatmap of val loss (mlp_mult vs topk)
# -------------------------------------------------------------------
loss_matrix = np.zeros((len(mlp_mults), len(topks)))
for i, mlp_mult in enumerate(mlp_mults):
    for j, topk in enumerate(topks):
        loss_matrix[i, j] = data.get((mlp_mult, topk), np.nan)

fig1, ax1 = plt.subplots(figsize=(8, 8))
im1 = ax1.imshow(loss_matrix, cmap=red_pink_cmap, aspect="auto")
ax1.set_xticks(range(len(topks)))
ax1.set_yticks(range(len(mlp_mults)))
ax1.set_xticklabels([str(k) if k is not None else "None" for k in topks], fontsize=15)
ax1.set_yticklabels(mlp_mults, fontsize=15)
ax1.set_xlabel("Top-K Value", fontsize=15)
ax1.set_ylabel("MLP Multiplier", fontsize=15)
# Дополнительно задаём размер тиков через tick_params (на случай, если set_* не охватит все)
ax1.tick_params(axis='both', labelsize=15)

for i in range(len(mlp_mults)):
    for j in range(len(topks)):
        val = loss_matrix[i, j]
        if not np.isnan(val):
            ax1.text(j, i, f"{val:.3f}", ha="center", va="center", color="white", fontsize=15)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Plot 2: Line plot (Top-K vs val loss, lines = mlp_mult)
# -------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(8, 8))
for idx, mlp_mult in enumerate(mlp_mults):
    losses = [data.get((mlp_mult, topk), np.nan) for topk in topks]
    topk_labels = [str(k) if k is not None else "None" for k in topks]
    color = palette[idx % len(palette)]
    ax2.plot(topk_labels, losses, marker="o", color=color, label=f"mlp_mult={mlp_mult}")
ax2.set_xlabel("Top-K Value", fontsize=15)
ax2.set_ylabel("Final Validation Loss", fontsize=15)
ax2.legend(fontsize=15, loc="best")
ax2.grid(True, alpha=0.3)
# Установка размера шрифта для меток осей
ax2.tick_params(axis='both', labelsize=15)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Plot 3: Heatmap of dead neurons proportion (тот же colormap, reversed)
# -------------------------------------------------------------------
if dead_neurons_data:
    dead_matrix = np.zeros((len(mlp_mults), len(topks)))
    for i, mlp_mult in enumerate(mlp_mults):
        for j, topk in enumerate(topks):
            dead_matrix[i, j] = dead_neurons_data.get((mlp_mult, topk), np.nan)

    fig3, ax3 = plt.subplots(figsize=(8, 8))
    im3 = ax3.imshow(dead_matrix, cmap=red_pink_cmap.reversed(), aspect="auto")
    ax3.set_xticks(range(len(topks)))
    ax3.set_yticks(range(len(mlp_mults)))
    ax3.set_xticklabels([str(k) if k is not None else "None" for k in topks], fontsize=15)
    ax3.set_yticklabels(mlp_mults, fontsize=15)
    ax3.set_xlabel("Top-K Value", fontsize=15)
    ax3.set_ylabel("MLP Multiplier", fontsize=15)
    ax3.tick_params(axis='both', labelsize=15)

    vmin, vmax = np.nanmin(dead_matrix), np.nanmax(dead_matrix)
    for i in range(len(mlp_mults)):
        for j in range(len(topks)):
            val = dead_matrix[i, j]
            if not np.isnan(val):
                norm_val = (val - vmin) / (vmax - vmin) if vmax != vmin else 0.5
                text_color = 'white' if norm_val > 0.5 else 'black'
                ax3.text(j, i, f"{val:.3f}", ha="center", va="center", color=text_color, fontsize=15)

    plt.tight_layout()
    plt.show()

# -------------------------------------------------------------------
# Plot 4: Line plot (MLP multiplier vs val loss, lines = Top-K) с log2 по X
# -------------------------------------------------------------------
fig4, ax4 = plt.subplots(figsize=(8,8))
for idx, topk in enumerate(topks):
    losses = [data.get((mlp_mult, topk), np.nan) for mlp_mult in mlp_mults]
    topk_label = str(topk) if topk is not None else "None"
    color = palette[idx % len(palette)]
    ax4.plot(mlp_mults, losses, marker="o", color=color, label=f"Top-K={topk_label}")

ax4.set_xscale('log', base=2)
ax4.set_xlabel("MLP Multiplier (log2)", fontsize=15)
ax4.set_ylabel("Final Validation Loss", fontsize=15)
ax4.legend(fontsize=15, loc="best")
ax4.grid(True, alpha=0.3, which="both")
ax4.set_xticks(mlp_mults)
ax4.set_xticklabels([str(m) for m in mlp_mults], fontsize=15)
ax4.tick_params(axis='y', labelsize=15)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Доп. график: Validation Loss vs Proportion of Activated Neurons (log scale)
# -------------------------------------------------------------------
if active_per_token_data:
    n_layer = 5
    hidden_dim = 256
    activated_proportion = {}
    for (mlp_mult, topk), active in active_per_token_data.items():
        total_neurons = n_layer * hidden_dim * mlp_mult
        activated_proportion[(mlp_mult, topk)] = active / total_neurons

    fig5, ax5 = plt.subplots(figsize=(8, 8))
    for idx, mlp_mult in enumerate(mlp_mults):
        x_vals = []
        y_vals = []
        for topk in topks:
            if (mlp_mult, topk) in activated_proportion and (mlp_mult, topk) in data:
                prop = activated_proportion[(mlp_mult, topk)]
                loss = data[(mlp_mult, topk)]
                if not np.isnan(loss) and prop > 0:
                    x_vals.append(prop)
                    y_vals.append(loss)
        if x_vals:
            color = palette[idx % len(palette)]
            ax5.plot(x_vals, y_vals, marker="o", color=color, label=f"mlp_mult={mlp_mult}",
                     linewidth=2, markersize=8)
    ax5.set_xscale('log')
    ax5.set_xlabel("Proportion of Activated Neurons (log10)", fontsize=15)
    ax5.set_ylabel("Final Validation Loss", fontsize=15)
    ax5.legend(fontsize=15, loc="best")
    ax5.grid(True, alpha=0.3, which="both")
    ax5.tick_params(axis='both', labelsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

# -------------------------------------------------------------------
# 1. Загрузка данных
# -------------------------------------------------------------------
import os
METRICS_PATH = "recomputed_metrics.pkl" if os.path.exists("recomputed_metrics.pkl") else "scaling/recomputed_metrics.pkl"
with open(METRICS_PATH, "rb") as f:
    results = pickle.load(f)

data = results["val_loss"]
dead_neurons_data = results["dead_proportion"]
active_per_token_data = results["active_per_token"]

mlp_mults = sorted(set(m for (m, _) in data.keys()))
topks = sorted(set(k for (_, k) in data.keys()), key=lambda x: (x is None, x))

# -------------------------------------------------------------------
# 2. Вспомогательные функции для осветления/затемнения цвета
# -------------------------------------------------------------------
def lighten_color(color, factor=0.5):
    rgb = np.array(mcolors.to_rgb(color))
    white = np.array([1, 1, 1])
    return mcolors.to_hex(rgb + (white - rgb) * factor)

def darken_color(color, factor=0.5):
    rgb = np.array(mcolors.to_rgb(color))
    black = np.array([0, 0, 0])
    return mcolors.to_hex(rgb + (black - rgb) * factor)

base_red = "#8a3434"
red_pink_cmap = LinearSegmentedColormap.from_list(
    "red_pink",
    [darken_color(base_red, 0.7), base_red, lighten_color(base_red, 0.8)],
    N=256
)

palette = [
    "#5f5a4a", "#6f6754", "#7f745e", "#8f8068", "#9f8d72",
    "#8a3434", "#984444", "#a65454", "#b46464", "#c27474",
]

# -------------------------------------------------------------------
# Plot 1: Heatmap of val loss (mlp_mult vs topk)
# -------------------------------------------------------------------
loss_matrix = np.zeros((len(mlp_mults), len(topks)))
for i, mlp_mult in enumerate(mlp_mults):
    for j, topk in enumerate(topks):
        loss_matrix[i, j] = data.get((mlp_mult, topk), np.nan)

fig1, ax1 = plt.subplots(figsize=(6, 6))
im1 = ax1.imshow(loss_matrix, cmap=red_pink_cmap, aspect="auto")
ax1.set_xticks(range(len(topks)))
ax1.set_yticks(range(len(mlp_mults)))
ax1.set_xticklabels([str(k) if k is not None else "None" for k in topks])
ax1.set_yticklabels(mlp_mults)
ax1.set_xlabel("Top-K Value")
ax1.set_ylabel("MLP Multiplier")
for i in range(len(mlp_mults)):
    for j in range(len(topks)):
        val = loss_matrix[i, j]
        if not np.isnan(val):
            ax1.text(j, i, f"{val:.3f}", ha="center", va="center", color="white", fontsize=15)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Plot 2: Line plot (Top-K vs val loss, lines = mlp_mult)
# -------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(6, 6))
for idx, mlp_mult in enumerate(mlp_mults):
    losses = [data.get((mlp_mult, topk), np.nan) for topk in topks]
    topk_labels = [str(k) if k is not None else "None" for k in topks]
    color = palette[idx % len(palette)]
    ax2.plot(topk_labels, losses, marker="o", color=color, label=f"mlp_mult={mlp_mult}")
ax2.set_xlabel("Top-K Value")
ax2.set_ylabel("Final Validation Loss")
ax2.legend(fontsize=15, loc="best")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Plot 3: Heatmap of dead neurons proportion (тот же colormap)
# -------------------------------------------------------------------
if dead_neurons_data:
    dead_matrix = np.zeros((len(mlp_mults), len(topks)))
    for i, mlp_mult in enumerate(mlp_mults):
        for j, topk in enumerate(topks):
            dead_matrix[i, j] = dead_neurons_data.get((mlp_mult, topk), np.nan)

    fig3, ax3 = plt.subplots(figsize=(6, 6))
    im3 = ax3.imshow(dead_matrix, cmap=red_pink_cmap.reversed(), aspect="auto")
    ax3.set_xticks(range(len(topks)))
    ax3.set_yticks(range(len(mlp_mults)))
    ax3.set_xticklabels([str(k) if k is not None else "None" for k in topks])
    ax3.set_yticklabels(mlp_mults)
    ax3.set_xlabel("Top-K Value")
    ax3.set_ylabel("MLP Multiplier")

    vmin, vmax = np.nanmin(dead_matrix), np.nanmax(dead_matrix)
    for i in range(len(mlp_mults)):
        for j in range(len(topks)):
            val = dead_matrix[i, j]
            if not np.isnan(val):
                norm_val = (val - vmin) / (vmax - vmin) if vmax != vmin else 0.5
                text_color = 'white' if norm_val > 0.5 else 'black'
                ax3.text(j, i, f"{val:.3f}", ha="center", va="center", color=text_color, fontsize=15)
    plt.tight_layout()
    plt.show()

# -------------------------------------------------------------------
# Plot 4: Line plot (MLP multiplier vs val loss, lines = Top-K) с log2 по X
# -------------------------------------------------------------------
fig4, ax4 = plt.subplots(figsize=(6.5, 6))
for idx, topk in enumerate(topks):
    losses = [data.get((mlp_mult, topk), np.nan) for mlp_mult in mlp_mults]
    topk_label = str(topk) if topk is not None else "None"
    color = palette[idx % len(palette)]
    ax4.plot(mlp_mults, losses, marker="o", color=color, label=f"Top-K={topk_label}")
ax4.set_xscale('log', base=2)
ax4.set_xlabel("MLP Multiplier (log2)")
ax4.set_ylabel("Final Validation Loss")
ax4.legend(fontsize=6, loc="best")
ax4.grid(True, alpha=0.3, which="both")
ax4.set_xticks(mlp_mults)
ax4.set_xticklabels([str(m) for m in mlp_mults])
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# Доп. график: Validation Loss vs Proportion of Activated Neurons (log scale)
# -------------------------------------------------------------------
if active_per_token_data:
    n_layer = 5
    hidden_dim = 256
    activated_proportion = {}
    for (mlp_mult, topk), active in active_per_token_data.items():
        total_neurons = n_layer * hidden_dim * mlp_mult
        activated_proportion[(mlp_mult, topk)] = active / total_neurons

    fig5, ax5 = plt.subplots(figsize=(6, 6))
    for idx, mlp_mult in enumerate(mlp_mults):
        x_vals = []
        y_vals = []
        for topk in topks:
            if (mlp_mult, topk) in activated_proportion and (mlp_mult, topk) in data:
                prop = activated_proportion[(mlp_mult, topk)]
                loss = data[(mlp_mult, topk)]
                if not np.isnan(loss) and prop > 0:
                    x_vals.append(prop)
                    y_vals.append(loss)
        if x_vals:
            color = palette[idx % len(palette)]
            ax5.plot(x_vals, y_vals, marker="o", color=color, label=f"mlp_mult={mlp_mult}",
                     linewidth=2, markersize=8)
    ax5.set_xscale('log')
    ax5.set_xlabel("Proportion of Activated Neurons (log10)", fontsize=12)
    ax5.set_ylabel("Final Validation Loss", fontsize=12)
    ax5.legend(fontsize=10, loc="best")
    ax5.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()

# =============================================================================
# СТЕПЕННАЯ МОДЕЛЬ В ИСХОДНОЙ ШКАЛЕ (нелинейная регрессия)
# =============================================================================

# Собираем обучающие данные: y = loss, x1 = mlp_mult, x2 = эффективный top_k
mlp_vals = []
k_eff_vals = []
loss_vals = []

for (mp, tk), loss in data.items():
    if loss is None or np.isnan(loss):
        continue
    # Определяем эффективное K
    if tk is None:
        if (mp, tk) in active_per_token_data:
            k_eff = active_per_token_data[(mp, tk)]
        else:
            continue
    else:
        k_eff = tk
    mlp_vals.append(mp)
    k_eff_vals.append(k_eff)
    loss_vals.append(loss)

mlp_arr = np.array(mlp_vals)
k_arr = np.array(k_eff_vals)
loss_arr = np.array(loss_vals)

# Определяем целевую функцию: Loss = C * mlp_mult^(-alpha) * top_k^(-beta)
def power_law(X, C, alpha, beta):
    mlp, k = X
    return C * (mlp ** (-alpha)) * (k ** (-beta))

# Для устойчивой подгонки нужны хорошие начальные параметры.
# Сначала получим их через логарифмическую линеаризацию (метод наименьших квадратов в логарифмическом пространстве)
log_mlp = np.log(mlp_arr)
log_k = np.log(k_arr)
log_loss = np.log(loss_arr)
X_log = np.column_stack([log_mlp, log_k])
from sklearn.linear_model import LinearRegression
lr_init = LinearRegression(fit_intercept=True)
lr_init.fit(X_log, log_loss)
log_C_init = lr_init.intercept_
alpha_init = -lr_init.coef_[0]
beta_init = -lr_init.coef_[1]
C_init = np.exp(log_C_init)
p0 = [C_init, alpha_init, beta_init]   # начальные приближения

# Нелинейная регрессия
try:
    popt, pcov = curve_fit(power_law, (mlp_arr, k_arr), loss_arr, p0=p0, maxfev=10000)
    C_opt, alpha_opt, beta_opt = popt
    # Предсказания
    pred_loss = power_law((mlp_arr, k_arr), C_opt, alpha_opt, beta_opt)
    # R² в исходной шкале
    r2_original = r2_score(loss_arr, pred_loss)
    
    print("=== СТЕПЕННАЯ МОДЕЛЬ (нелинейная регрессия в исходной шкале) ===")
    print(f"Loss = {C_opt:.6f} * mlp_mult^(-{alpha_opt:.6f}) * top_k^(-{beta_opt:.6f})")
    print(f"R² (original scale) = {r2_original:.6f}")
except Exception as e:
    print("Не удалось выполнить нелинейную регрессию:", e)
    # Если не удалось, используем значения из линейной регрессии на логарифмах как запасной вариант
    C_opt, alpha_opt, beta_opt = C_init, alpha_init, beta_init
    pred_loss = C_opt * (mlp_arr ** (-alpha_opt)) * (k_arr ** (-beta_opt))
    r2_original = r2_score(loss_arr, pred_loss)
    print("(используются оценки из логарифмической регрессии)")
    print(f"Loss = {C_opt:.6f} * mlp_mult^(-{alpha_opt:.6f}) * top_k^(-{beta_opt:.6f})")
    print(f"R² (original scale) = {r2_original:.6f}")

# -------------------------------------------------------------------
# График: истина против предсказания (исходная шкала)
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(loss_arr, pred_loss, alpha=1.0, color="#8a3434", s=40, label='All data')
lims = [min(loss_arr.min(), pred_loss.min()), max(loss_arr.max(), pred_loss.max())]
ax.plot(lims, lims, 'k--', alpha=0.8, label='$y = x$')

formula = r'$\mathrm{Loss} = %.3f \cdot \mathrm{mlp\_mult}^{ -%.3f } \cdot \mathrm{top\_k}^{ -%.3f }$' % (C_opt, alpha_opt, beta_opt)
r2_text = r'$R^2 = %.3f$' % r2_original
ax.text(0.05, 0.95, formula + '\n' + r2_text, transform=ax.transAxes, fontsize=15,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel("True Validation Loss", fontsize=15)
ax.set_ylabel("Prediction (power law)", fontsize=15)
ax.legend(loc='best', fontsize=15)
ax.tick_params(axis='both', labelsize=15)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =============================================================================
# Тепловые карты: реальный loss, предсказание, разница (в исходной шкале)
# =============================================================================

# Перестраиваем матрицы для всех комбинаций mlp_mult и topk
mlp_mults_sorted = sorted(set(m for (m, _) in data.keys()))
topks_sorted = sorted(set(k for (_, k) in data.keys()), key=lambda x: (x is None, x))

real_matrix = np.zeros((len(mlp_mults_sorted), len(topks_sorted)))
pred_matrix = np.zeros((len(mlp_mults_sorted), len(topks_sorted)))
diff_matrix = np.zeros((len(mlp_mults_sorted), len(topks_sorted)))

for i, mlp_mult in enumerate(mlp_mults_sorted):
    for j, topk in enumerate(topks_sorted):
        real = data.get((mlp_mult, topk), np.nan)
        real_matrix[i, j] = real
        if not np.isnan(real):
            # Определяем эффективное K для предсказания
            if topk is None:
                k_eff = active_per_token_data.get((mlp_mult, topk), np.nan)
                if np.isnan(k_eff):
                    pred = np.nan
                else:
                    pred = C_opt * (mlp_mult ** (-alpha_opt)) * (k_eff ** (-beta_opt))
            else:
                pred = C_opt * (mlp_mult ** (-alpha_opt)) * (topk ** (-beta_opt))
            pred_matrix[i, j] = pred
            diff_matrix[i, j] = real - pred
        else:
            pred_matrix[i, j] = np.nan
            diff_matrix[i, j] = np.nan

# Цветовая карта для разницы (расходящаяся)
div_cmap = plt.cm.coolwarm

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Реальный loss
im1 = axes[0].imshow(real_matrix, cmap=red_pink_cmap, aspect="auto")
axes[0].set_xticks(range(len(topks_sorted)))
axes[0].set_yticks(range(len(mlp_mults_sorted)))
axes[0].set_xticklabels([str(k) if k is not None else "None" for k in topks_sorted])
axes[0].set_yticklabels(mlp_mults_sorted)
axes[0].set_xlabel("Top-K Value")
axes[0].set_ylabel("MLP Multiplier")
axes[0].set_title("True Validation Loss")
vmin_real, vmax_real = np.nanmin(real_matrix), np.nanmax(real_matrix)
for i in range(len(mlp_mults_sorted)):
    for j in range(len(topks_sorted)):
        val = real_matrix[i, j]
        if not np.isnan(val):
            norm_val = (val - vmin_real) / (vmax_real - vmin_real) if vmax_real != vmin_real else 0.5
            text_color = 'white' if norm_val > 0.5 else 'black'
            axes[0].text(j, i, f"{val:.3f}", ha="center", va="center", color=text_color, fontsize=9)

# 2. Предсказанный loss
im2 = axes[1].imshow(pred_matrix, cmap=red_pink_cmap, aspect="auto")
axes[1].set_xticks(range(len(topks_sorted)))
axes[1].set_yticks(range(len(mlp_mults_sorted)))
axes[1].set_xticklabels([str(k) if k is not None else "None" for k in topks_sorted])
axes[1].set_yticklabels(mlp_mults_sorted)
axes[1].set_xlabel("Top-K Value")
axes[1].set_ylabel("MLP Multiplier")
axes[1].set_title("Predicted Loss (Power Law)")
vmin_pred, vmax_pred = np.nanmin(pred_matrix), np.nanmax(pred_matrix)
for i in range(len(mlp_mults_sorted)):
    for j in range(len(topks_sorted)):
        val = pred_matrix[i, j]
        if not np.isnan(val):
            norm_val = (val - vmin_pred) / (vmax_pred - vmin_pred) if vmax_pred != vmin_pred else 0.5
            text_color = 'white' if norm_val > 0.5 else 'black'
            axes[1].text(j, i, f"{val:.3f}", ha="center", va="center", color=text_color, fontsize=9)

# 3. Разница
max_abs_diff = max(abs(np.nanmin(diff_matrix)), abs(np.nanmax(diff_matrix)))
im3 = axes[2].imshow(diff_matrix, cmap=div_cmap, aspect="auto", vmin=-max_abs_diff, vmax=max_abs_diff)
axes[2].set_xticks(range(len(topks_sorted)))
axes[2].set_yticks(range(len(mlp_mults_sorted)))
axes[2].set_xticklabels([str(k) if k is not None else "None" for k in topks_sorted])
axes[2].set_yticklabels(mlp_mults_sorted)
axes[2].set_xlabel("Top-K Value")
axes[2].set_ylabel("MLP Multiplier")
axes[2].set_title("Residual (True - Predicted)")
for i in range(len(mlp_mults_sorted)):
    for j in range(len(topks_sorted)):
        val = diff_matrix[i, j]
        if not np.isnan(val):
            norm_val = (val + max_abs_diff) / (2 * max_abs_diff) if max_abs_diff > 0 else 0.5
            text_color = 'white' if norm_val < 0.3 or norm_val > 0.7 else 'black'
            axes[2].text(j, i, f"{val:.3f}", ha="center", va="center", color=text_color, fontsize=9)

plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
# Log-space OLS fit of L = c * M^alpha * K^beta (the estimates reported in the paper).
# For K=None the measured number of active neurons per token is used as the effective K.
lr, X = lr_init, X_log
C, alpha, beta = C_init, alpha_init, beta_init
r2 = r2_score(loss_arr, C * mlp_arr ** (-alpha) * k_arr ** (-beta))
Xdata = np.vstack([mlp_arr, k_arr])
print(f"log-OLS: c={C:.4f}, alpha={-alpha:.4f}, beta={-beta:.4f}, R^2={r2:.4f}")


In [ ]:
# ----- Исходная модель на всех данных (как у вас) -----
# ... (ваш код до вычисления C, alpha, beta, r2)

print("=== Модель на ВСЕХ данных ===")
print(f"Loss = {C:.6f} * mlp_mult^(-{alpha:.6f}) * top_k^(-{beta:.6f})")
print(f"R² = {r2:.6f}\n")

# ----- Модель без (64,8) и (64,16) -----
exclude_pairs = {(64, 8), (64, 16)}
mlp_list2 = []
k_eff_list2 = []
loss_list2 = []

for (mp, tk), loss in data.items():
    if loss is None or np.isnan(loss):
        continue
    if (mp, tk) in exclude_pairs:
        continue
    if tk is None:
        if (mp, tk) in active_per_token_data:
            k_eff = active_per_token_data[(mp, tk)]
            mlp_list2.append(mp)
            k_eff_list2.append(k_eff)
            loss_list2.append(loss)
    else:
        mlp_list2.append(mp)
        k_eff_list2.append(tk)
        loss_list2.append(loss)

mlp_arr2 = np.array(mlp_list2)
k_arr2 = np.array(k_eff_list2)
loss_arr2 = np.array(loss_list2)

log_mlp2 = np.log(mlp_arr2)
log_k2 = np.log(k_arr2)
log_loss2 = np.log(loss_arr2)
X2 = np.column_stack([log_mlp2, log_k2])
lr2 = LinearRegression(fit_intercept=True)
lr2.fit(X2, log_loss2)

log_C2 = lr2.intercept_
alpha2 = -lr2.coef_[0]
beta2 = -lr2.coef_[1]
C2 = np.exp(log_C2)

pred_loss2 = C2 * (mlp_arr2 ** (-alpha2)) * (k_arr2 ** (-beta2))
r2_2 = r2_score(loss_arr2, pred_loss2)

print("=== Модель без (64,8) и (64,16) ===")
print(f"Loss = {C2:.6f} * mlp_mult^(-{alpha2:.6f}) * top_k^(-{beta2:.6f})")
print(f"R² = {r2_2:.6f}")

# Можно также вычислить относительное изменение коэффициентов
print(f"\nИзменение alpha: {(alpha2 - alpha)/alpha*100:.2f}%")
print(f"Изменение beta:  {(beta2 - beta)/beta*100:.2f}%")
print(f"Изменение R²:    {(r2_2 - r2)/r2*100:.2f}%")

In [ ]:
from scipy import stats

# --- Доверительные интервалы для коэффициентов ---
# Стандартные ошибки коэффициентов
n = len(log_loss)                     # число точек
p = X.shape[1]                        # число предикторов (2: log_mlp, log_k)
dof = n - p - 1                       # степени свободы
# Остаточная дисперсия
mse = np.sum((log_loss - lr.predict(X))**2) / dof
# Ковариационная матрица коэффициентов
X_with_intercept = np.column_stack([np.ones(n), X])
cov_full = np.linalg.inv(X_with_intercept.T @ X_with_intercept) * mse
se_coef = np.sqrt(np.diag(cov_full))[1:]  # standard errors of (log_mlp, log_k), intercept included in the design

# Коэффициенты регрессии (raw coefficients for log_mlp, log_k)
raw_coef = lr.coef_   # order: for log_mlp, log_k
# Для α и β: α = -coef_log_mlp, β = -coef_log_k
alpha_hat = -raw_coef[0]
beta_hat = -raw_coef[1]
se_alpha = se_coef[0]   # так как α = -coef, то стандартная ошибка та же
se_beta = se_coef[1]

# t-критическое значение для 95% ДИ
t_crit = stats.t.ppf(0.975, dof)

alpha_ci = (alpha_hat - t_crit * se_alpha, alpha_hat + t_crit * se_alpha)
beta_ci = (beta_hat - t_crit * se_beta, beta_hat + t_crit * se_beta)

# Доверительный интервал для ln(C) и затем для C
log_C_hat = lr.intercept_
se_log_C = np.sqrt(mse * (1/n + np.mean(X, axis=0).reshape(1, -1) @ np.linalg.inv(X.T @ X) @ np.mean(X, axis=0).reshape(-1,1)))
# Упрощённо: стандартная ошибка intercept из ковариационной матрицы
# Добавим intercept в матрицу X
X_with_intercept = np.column_stack([np.ones(n), X])
cov_full = np.linalg.inv(X_with_intercept.T @ X_with_intercept) * mse
se_log_C_full = np.sqrt(cov_full[0, 0])
log_C_ci = (log_C_hat - t_crit * se_log_C_full, log_C_hat + t_crit * se_log_C_full)
C_ci = (np.exp(log_C_ci[0]), np.exp(log_C_ci[1]))

# --- Вывод результатов ---
print("\n" + "="*60)
print("Доверительные интервалы (95%) для параметров степенной модели")
print("="*60)
print(f"α (показатель при mlp_mult): {alpha_hat:.6f}  ДИ: [{alpha_ci[0]:.6f}, {alpha_ci[1]:.6f}]")
print(f"β (показатель при top_k):    {beta_hat:.6f}   ДИ: [{beta_ci[0]:.6f}, {beta_ci[1]:.6f}]")
print(f"C (константа):               {C:.6f}       ДИ: [{C_ci[0]:.6f}, {C_ci[1]:.6f}]")
print(f"Число наблюдений: {n}, степени свободы: {dof}")
print("="*60 + "\n")

# --- Доверительный интервал для предсказанного loss (при конкретных параметрах) ---
# Пример: для mlp_mult=1, top_k=32 (или для средних значений)
def predict_with_ci(mlp_val, k_val, alpha, beta, C, lr_model, X_data, log_loss_data, confidence=0.95):
    """
    Возвращает предсказанное значение loss и его доверительный интервал.
    Для новой точки (mlp, k) вычисляем log_pred = ln(C) - α ln(mlp) - β ln(k).
    Доверительный интервал для среднего предсказания (не для отдельного наблюдения).
    """
    log_k_eff = np.log(k_val) if k_val > 0 else 0
    log_mlp = np.log(mlp_val)
    # Вход для модели: [log_mlp, log_k]
    X_new = np.array([[log_mlp, log_k_eff]])
    log_pred = lr_model.predict(X_new)[0]
    # Стандартная ошибка предсказания среднего
    # Добавим intercept
    X_new_with_intercept = np.column_stack([np.ones(1), X_new])
    se_pred = np.sqrt(X_new_with_intercept @ cov_full @ X_new_with_intercept.T)[0,0]
    t_crit_local = stats.t.ppf((1 + confidence) / 2, dof)
    interval = (np.exp(log_pred - t_crit_local * se_pred), np.exp(log_pred + t_crit_local * se_pred))
    return np.exp(log_pred), interval

In [ ]:
# =============================================================================
# МОДЕЛЬ E: Loss = C1 * mlp^(-α) + C2 * (mlp / topk)^γ
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.utils import resample
from sklearn.metrics import r2_score

# Данные уже собраны в mlp_arr, k_arr, loss_arr (из предыдущего блока)
# Определяем модель E
def model_E(x, C1, alpha, C2, gamma):
    mlp, topk = x
    return C1 * (mlp ** (-alpha)) + C2 * (mlp / topk) ** gamma

# Подгонка на всех данных
popt_E, _ = curve_fit(lambda x, *p: model_E(x, *p), Xdata, loss_arr,
                      p0=[1.0, 0.1, 0.1, 0.1], maxfev=5000)
C1_E, alpha_E, C2_E, gamma_E = popt_E
pred_E = model_E(Xdata, C1_E, alpha_E, C2_E, gamma_E)
r2_E = r2_score(loss_arr, pred_E)

print("=== Модель E: Loss = C1 * mlp^(-α) + C2 * (mlp/topk)^γ ===")
print(f"C1   = {C1_E:.6f}")
print(f"α    = {alpha_E:.6f}")
print(f"C2   = {C2_E:.6f}")
print(f"γ    = {gamma_E:.6f}")
print(f"R²   = {r2_E:.6f}")

# Бутстрап для доверительных интервалов (опционально)
n_bootstrap = 1000
n_points = len(loss_arr)
boot_params_E = []
np.random.seed(42)
for _ in range(n_bootstrap):
    idx = resample(np.arange(n_points), replace=True, n_samples=n_points)
    X_boot = Xdata[:, idx]
    y_boot = loss_arr[idx]
    try:
        popt_boot, _ = curve_fit(lambda x, *p: model_E(x, *p), X_boot, y_boot,
                                 p0=popt_E, maxfev=5000)
        boot_params_E.append(popt_boot)
    except:
        continue
boot_params_E = np.array(boot_params_E)
lower_E = np.percentile(boot_params_E, 2.5, axis=0)
upper_E = np.percentile(boot_params_E, 97.5, axis=0)

print("\n95% доверительные интервалы (бутстрап):")
for name, val, low, high in zip(['C1','alpha','C2','gamma'], popt_E, lower_E, upper_E):
    print(f"{name:4s} = {val:.6f}  [{low:.6f}, {high:.6f}]")

# График
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(loss_arr, pred_E, alpha=0.7, color="#8a3434", s=40)
lims = [min(loss_arr.min(), pred_E.min()), max(loss_arr.max(), pred_E.max())]
ax.plot(lims, lims, 'k--', alpha=0.5, label='$y = x$')
formula = r'$\mathrm{Loss} = %.5f \cdot \mathrm{mlp}^{ -%.5f } + %.5f \cdot (\mathrm{mlp}/\mathrm{topk})^{ %.5f }$' % (C1_E, alpha_E, C2_E, gamma_E)
r2_text = r'$R^2 = %.6f$' % r2_E
ax.text(0.05, 0.95, formula + '\n' + r2_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.set_xlabel("True Validation Loss")
ax.set_ylabel("Validation Loss Prediction")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rank structure of the validation-loss matrix and alternative functional forms
finite_topks = [k for k in topks if k is not None]
for name, ks in [("7x4 (finite K)", finite_topks), ("7x5 (incl. K=None)", topks)]:
    mat = np.array([[data[(m, k)] for k in ks] for m in mlp_mults])
    s = np.linalg.svd(mat, compute_uv=False)
    print(f"{name}: sigma1={s[0]:.3f}, sigma2={s[1]:.3f}")

A_lin = np.column_stack([np.ones_like(mlp_arr, dtype=float), mlp_arr, k_arr])
A_log = np.column_stack([np.ones_like(mlp_arr, dtype=float), np.log(mlp_arr), np.log(k_arr)])
for name, A in [("linear  L = a + b*M + c*K", A_lin), ("log     L = a + b*lnM + c*lnK", A_log)]:
    coef, *_ = np.linalg.lstsq(A, loss_arr, rcond=None)
    print(f"{name}: R^2={r2_score(loss_arr, A @ coef):.4f}")
print(f"power   L = c*M^alpha*K^beta: R^2={r2:.4f}")
